# Stan in Colab

Fitting a graphical model with **Stan**, through the `mcmc` command line tool.

Set the model and the sampler below, then run every cell in order. The notebook fits the model, checks that it converged, draws the posterior, and packages the run so it can be opened in the report app.

## Model

In the editor, open the **Run** tab and press **Copy graph**.
Paste it between the quotes below, then run every cell in order.

In [ ]:
GRAPH = r'''

'''

import json

if not GRAPH.strip():
    raise SystemExit("Paste your graph above, then run this cell again.")

with open("model.json", "w") as f:
    f.write(GRAPH)

print("model:", json.loads(GRAPH)["name"])

## Settings

Change these and run again from here.

In [ ]:
CHAINS = 2
DRAWS = 1000
WARMUP = 1000
SEED = 42

## Install

`mcmc` is a self-contained binary. `mcmc setup` then installs the Stan toolchain, which takes a few minutes on a fresh Colab runtime.

In [ ]:
!curl -fsSL https://mcmcjs.github.io/install.sh | sh

import os

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!mcmc --version

In [ ]:
!mcmc setup --engine stan

## The Stan model

What the graph becomes as code, and the spec that runs it.

In [ ]:
!mcmc convert model.json --stan
!cat model.stan

## Fit

`mcmc run` samples, checks convergence, and records the run.

In [ ]:
seed_flag = f" --seed {SEED}" if SEED is not None else ""

!mcmc run model.toml --chains {CHAINS} --draws {DRAWS} --warmup {WARMUP}{seed_flag}

## Did it converge?

R-hat near 1 and a healthy effective sample size for every parameter. `mcmc diagnose` exits non-zero when it did not, so read this before the posterior.

In [ ]:
!mcmc summary
!mcmc diagnose

## Plots

Traces and ranks show the chains mixing, densities and the forest plot show the posterior itself.

In [ ]:
from IPython.display import SVG, display

for kind in ["trace","density","forest","rank"]:
    !mcmc plot --kind {kind} --format svg -o {kind}.svg
    print(kind)
    display(SVG(f"{kind}.svg"))

## Open the run in the report app

A run bundle holds the samples, the spec and the diagnostics in one file. Download it and drop it into [the report app](https://mcmcjs.github.io/report/) to explore every parameter.

In [ ]:
!mcmc export bundle -o run.mcmcrun.json

from google.colab import files  # Colab only

files.download("run.mcmcrun.json")